In [8]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
from models import ContrastiveSpaceModel
import pickle
import numpy as np
import os
from train_utils import train_contrastive
from data_utils import ContrastiveCollator
import evaluation_utils as eval_utils

In [9]:
source_model = 'lstm'
source_key = f'{source_model}_embeddings'
batch_size = 4
device_name = 'cuda:0'
shared_dim = 64

val_path = 'data/contrastive_dataset/CA_test.pickle'

with open(val_path, 'rb') as f:
    val_dataset = pickle.load(f)

collator = ContrastiveCollator(pad_id=0)

source_dim = val_dataset[0][source_key].shape[0]
transformer_dim = val_dataset[0]['transformer_embeddings'].shape[0]

In [10]:
print(len(val_dataset))
print(val_dataset[0].keys())

756
dict_keys(['harmony_ids', 'pianoroll', 'lstm_embeddings', 'matrix', 'matrix_embeddings', 'bot', 'bot_embeddings', 'graph', 'graph_embeddings', 'transformer_embeddings'])


In [11]:
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

In [12]:
if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

model = ContrastiveSpaceModel(source_dim, transformer_dim, shared_dim=shared_dim)
model.to(device)

checkpoint = torch.load(f'saved_models/contrastive/{source_model}.pt', map_location=device_name)
model.load_state_dict(checkpoint)
model.eval()

RuntimeError: Error(s) in loading state_dict for ContrastiveSpaceModel:
	size mismatch for source_proj.net.2.weight: copying a param with shape torch.Size([32, 256]) from checkpoint, the shape in current model is torch.Size([64, 256]).
	size mismatch for source_proj.net.2.bias: copying a param with shape torch.Size([32]) from checkpoint, the shape in current model is torch.Size([64]).
	size mismatch for transformer_proj.net.2.weight: copying a param with shape torch.Size([32, 256]) from checkpoint, the shape in current model is torch.Size([64, 256]).
	size mismatch for transformer_proj.net.2.bias: copying a param with shape torch.Size([32]) from checkpoint, the shape in current model is torch.Size([64]).

In [6]:
Zs, Zt = eval_utils.collect_embeddings(model, val_loader, source_key, device)

In [7]:
print(Zs.shape, Zt.shape)

torch.Size([756, 64]) torch.Size([756, 64])


In [8]:
proc_error, Zs_aligned, Zt_centered = eval_utils.procrustes_error(Zs, Zt)
print("Procrustes error:", proc_error)

Procrustes error: 11.258919715881348


In [9]:
rsa_corr = eval_utils.distance_matrix_correlation(Zs, Zt)
print("Distance matrix correlation:", rsa_corr)

Distance matrix correlation: 0.8768659830093384


In [10]:
retrieval = eval_utils.retrieval_accuracy(Zs, Zt)
print(retrieval)

{'top1': 0.5238095238095238, 'top5': 0.8253968253968254}


In [11]:
eval_utils.evaluate_alignment(model, val_loader, source_key, device)

=== Alignment Evaluation ===
Procrustes error: 11.258919715881348
Distance matrix correlation: 0.8768659830093384
Retrieval accuracy: {'top1': 0.5238095238095238, 'top5': 0.8253968253968254}


{'procrustes_error': 11.258919715881348,
 'rsa_correlation': 0.8768659830093384,
 'retrieval': {'top1': 0.5238095238095238, 'top5': 0.8253968253968254}}